# 대피도 YOLO Round 3 AUTO — 무수동검수 버전

Round 2 `best.pt`에서 이어서 `exit / stair / you_are_here`를 보강합니다. **사람이 박스를 그리거나 검수하지 않습니다.** 대신 두 번의 모델 예측 일치도와 confidence/박스 크기 규칙으로 pseudo-label을 자동 필터링합니다.

⚠️ 이 모드의 val/test도 자동 pseudo-label이므로 최종 P/R/mAP은 **참고값**이며 사람 검수 ground truth 성능이 아닙니다.


In [ ]:
# 0. 환경 준비
!pip -q install ultralytics pillow numpy requests
from pathlib import Path
import shutil, zipfile, json, os
from google.colab import files, drive
from ultralytics import YOLO

KIT=Path('/content/evac_round3_auto_kit')
if not KIT.exists():
    print('evac_round3_auto_kit.zip을 업로드하세요.')
    up=files.upload()
    z=next(Path(k) for k in up if k.lower().endswith('.zip'))
    with zipfile.ZipFile(z) as f: f.extractall('/content')
print('KIT=',KIT,'exists=',KIT.exists())
NAMES=['exit','stair','elevator','extinguisher','hydrant','you_are_here','door','room']
SEED=KIT/'seed_model'/'best_round2.pt'
IMAGE_SIZE=960
BATCH=-1
assert SEED.exists(), SEED


## 1. 공개 라이선스 실제 대피도 자동 수집
기존 Round 1/2 source URL을 제외하고 기본 80장을 시도합니다.


In [ ]:
RAW=Path('/content/round3_auto_real_raw')
!python {KIT/'scripts'/'collect_round3_real.py'} --out {RAW} --target 80 --depth 6 --max-side 2200 --width 2200 --clean --exclude-csv {KIT/'manifests'/'exclude_prior_sources.csv'} --curated-csv {KIT/'manifests'/'curated_fresh_sources.csv'}
print('수집 이미지:',len(list((RAW/'images').glob('*.jpg'))))
assert len(list((RAW/'images').glob('*.jpg'))) >= 10, '수집 이미지가 너무 적습니다. 1번 셀을 다시 실행하세요.'


## 2. 핵심 클래스 후보 자동 분석
Round 2 모델로 `exit/stair/you_are_here` 후보 분포를 확인합니다. 이 단계는 라벨 정답 판정이 아니라 자동 선택 보조입니다.


In [ ]:
RANK=Path('/content/round3_auto_ranked')
!python {KIT/'scripts'/'rank_critical.py'} --model {SEED} --images {RAW/'images'} --metadata {RAW/'metadata.csv'} --out {RANK} --conf 0.06 --imgsz {IMAGE_SIZE} --top 80
print((RANK/'ranking_summary.json').read_text())


## 3. 자동검수 pseudo-label 생성
같은 이미지를 960/1280 두 해상도로 예측합니다. 같은 클래스가 비슷한 위치에 반복 검출된 박스를 우선 채택하고, 매우 높은 confidence의 단독 검출만 추가 채택합니다. **사람 작업 없음.**


In [ ]:
PSEUDO=Path('/content/round3_auto_pseudo')
!python {KIT/'scripts'/'auto_pseudo_review.py'} --model {SEED} --images {RAW/'images'} --out {PSEUDO} --imgsz-a 960 --imgsz-b 1280 --conf 0.06 --iou 0.42
print((PSEUDO/'pseudo_label_report.json').read_text())


## 4. 자동 train/val/test 분할
핵심 클래스 pseudo-label을 val/test에 가능한 만큼 분산합니다. 데이터가 부족하면 자동으로 목표 박스 수를 낮추고 경고를 남깁니다.


In [ ]:
DATA=Path('/content/round3_auto_dataset')
!python {KIT/'scripts'/'split_pseudo_auto.py'} --pseudo {PSEUDO} --out {DATA} --desired-critical 8
ROUND3_YAML=DATA/'data_round3.yaml'
print((DATA/'split_report.json').read_text())
print(ROUND3_YAML.read_text())


## 5. Google Drive 연결 + Stage A — 최대 15 epoch
GPU 런타임 권장. Drive에 `last.pt`가 있으면 자동 resume합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
from ultralytics import YOLO

# Google Drive 연결
drive.mount('/content/drive', force_remount=False)

# 경로 설정
KIT = Path("/content/evac_round3_auto_kit")
SEED = KIT / "seed_model" / "best_round2.pt"

ROUND3_YAML = Path("/content/round3_auto_dataset/data_round3.yaml")

# 메모리 절약 설정
IMAGE_SIZE = 768
BATCH = 2

print("SEED 존재:", SEED.exists())
print("YAML 존재:", ROUND3_YAML.exists())

assert SEED.exists(), f"SEED 모델 없음: {SEED}"
assert ROUND3_YAML.exists(), f"YAML 없음: {ROUND3_YAML}"

# Drive 체크포인트 저장 위치
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/evacuation_checkpoints/round3_auto"
)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

RUN_A = "evac_round3_auto_stage_a"
DIR_A = DRIVE_ROOT / RUN_A
LAST_A = DIR_A / "weights" / "last.pt"

# 기존 학습이 있으면 이어서
if LAST_A.exists():
    print("✅ 기존 Stage A 체크포인트 발견 -> 이어서 학습")
    model_a = YOLO(str(LAST_A))
    stage_a = model_a.train(resume=True)

# 없으면 새로 15 epoch
else:
    print("✅ Stage A 새 학습 시작: 15 epochs")

    model_a = YOLO(str(SEED))

    stage_a = model_a.train(
        data=str(ROUND3_YAML),

        epochs=15,
        patience=5,

        imgsz=768,
        batch=2,

        optimizer="AdamW",
        lr0=5e-4,
        lrf=0.05,
        weight_decay=5e-4,

        warmup_epochs=1.0,
        cos_lr=True,

        degrees=3.0,
        translate=0.05,
        scale=0.15,
        perspective=0.001,

        hsv_h=0.01,
        hsv_s=0.15,
        hsv_v=0.18,

        fliplr=0.0,
        flipud=0.0,

        mosaic=0.10,
        mixup=0.0,
        close_mosaic=5,

        amp=True,
        cache=False,
        workers=1,

        project=str(DRIVE_ROOT),
        name=RUN_A,
        exist_ok=True,

        seed=20260816,
    )

BEST_A = DIR_A / "weights" / "best.pt"
LAST_A = DIR_A / "weights" / "last.pt"

print("\n==============================")
print("Stage A 완료")
print("BEST_A 존재:", BEST_A.exists())
print("LAST_A 존재:", LAST_A.exists())
print("==============================")

## 6. Stage B — 최대 5 epoch
**Stage A가 끝난 다음** 실행하세요. 동시에 실행하지 않습니다.


In [ ]:
RUN_B='evac_round3_auto_stage_b'; DIR_B=DRIVE_ROOT/RUN_B; LAST_B=DIR_B/'weights'/'last.pt'
if LAST_B.exists():
    print('Stage B resume:',LAST_B); stage_b=YOLO(str(LAST_B)).train(resume=True)
else:
    assert BEST_A.exists(),'Stage A best.pt가 없습니다.'
    stage_b=YOLO(str(BEST_A)).train(data=str(ROUND3_YAML),epochs=5,patience=3,imgsz=IMAGE_SIZE,batch=BATCH,optimizer='AdamW',lr0=1e-4,lrf=.1,weight_decay=5e-4,warmup_epochs=.5,cos_lr=True,degrees=1.5,translate=.02,scale=.06,perspective=.0003,hsv_h=.003,hsv_s=.07,hsv_v=.08,fliplr=0,flipud=0,mosaic=0,mixup=0,amp=True,cache=False,workers=2,project=str(DRIVE_ROOT),name=RUN_B,exist_ok=True,seed=20260815)
BEST_B=DIR_B/'weights'/'best.pt'
print('BEST_B',BEST_B,BEST_B.exists())


## 7. Stage A/B validation 비교


In [ ]:
def val_score(p):
    r=YOLO(str(p)).val(data=str(ROUND3_YAML),split='val',imgsz=IMAGE_SIZE,verbose=False)
    return r,float(r.box.map50)+0.5*float(r.box.mr)
ra,sa=val_score(BEST_A); rb,sb=val_score(BEST_B)
print('A val',float(ra.box.mp),float(ra.box.mr),float(ra.box.map50),float(ra.box.map),'score',sa)
print('B val',float(rb.box.mp),float(rb.box.mr),float(rb.box.map50),float(rb.box.map),'score',sb)
BEST_ROUND3=BEST_B if sb>=sa else BEST_A
print('선택:',BEST_ROUND3)


## 8. pseudo-test 자동 참고 평가
**이 수치는 사람 검수 ground truth가 아닌 pseudo-label 기준 참고값입니다.**


In [ ]:
TARGET=Path('/content/round3_auto_target_result.json')
!python {KIT/'scripts'/'evaluate_round3_auto.py'} --model {BEST_ROUND3} --data {ROUND3_YAML} --imgsz {IMAGE_SIZE} --out {TARGET}
print(TARGET.read_text())


## 9. 최종 내보내기


In [ ]:
from pathlib import Path
from ultralytics import YOLO
from google.colab import files
import shutil

# =========================
# 9. 최종 모델 내보내기
# =========================

# 경로를 다시 직접 정의
DATA = Path("/content/round3_auto_dataset")
PSEUDO = Path("/content/round3_auto_pseudo")
ROUND3_YAML = DATA / "data_round3.yaml"
TARGET = Path("/content/round3_auto_target_result.json")

# 메모리 절약 설정과 동일하게
IMAGE_SIZE = 768

# BEST_ROUND3 변수가 없는 경우 Drive에서 자동 탐색
try:
    BEST_ROUND3
    BEST_ROUND3 = Path(BEST_ROUND3)
except NameError:
    drive_root = Path(
        "/content/drive/MyDrive/evacuation_checkpoints/round3_auto"
    )

    candidates = [
        drive_root / "evac_round3_auto_stage_b" / "weights" / "best.pt",
        drive_root / "evac_round3_auto_stage_a" / "weights" / "best.pt",
    ]

    BEST_ROUND3 = next(
        (p for p in candidates if p.exists()),
        None
    )

assert BEST_ROUND3 is not None, "최종 best.pt를 찾지 못했습니다."
assert Path(BEST_ROUND3).exists(), f"모델 없음: {BEST_ROUND3}"

print("최종 모델:", BEST_ROUND3)
print("YAML 존재:", ROUND3_YAML.exists())
print("평가 JSON 존재:", TARGET.exists())
print("split report 존재:", (DATA / "split_report.json").exists())
print(
    "pseudo report 존재:",
    (PSEUDO / "pseudo_label_report.json").exists()
)

# -------------------------
# 1. ONNX 변환
# -------------------------

FINAL = YOLO(str(BEST_ROUND3))

onnx_path = FINAL.export(
    format="onnx",
    imgsz=IMAGE_SIZE,
    opset=12,
    simplify=True
)

onnx_path = Path(onnx_path)

# -------------------------
# 2. 최종 출력 폴더
# -------------------------

OUT = Path("/content/evac_model_round3_auto_final")

shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True, exist_ok=True)

# -------------------------
# 3. 필요한 결과 복사
# -------------------------

copy_items = [
    (Path(BEST_ROUND3), "best.pt"),
    (onnx_path, "best.onnx"),
    (ROUND3_YAML, "data_round3_auto.yaml"),
    (TARGET, "target_result_pseudo.json"),
    (DATA / "split_report.json", "split_report.json"),
    (
        PSEUDO / "pseudo_label_report.json",
        "pseudo_label_report.json"
    ),
]

for src, dst_name in copy_items:
    src = Path(src)

    if src.exists():
        shutil.copy2(src, OUT / dst_name)
        print("✅ 복사:", dst_name)
    else:
        print("⚠️ 없음, 건너뜀:", src)

# -------------------------
# 4. 주의사항 기록
# -------------------------

important_text = """AUTO MODE

val/test labels are pseudo-labels,
not human-verified ground truth.

Precision, Recall, mAP50 and mAP50-95
are reference-only metrics.

These metrics must not be treated as
true human-ground-truth accuracy.
"""

(OUT / "IMPORTANT.txt").write_text(
    important_text,
    encoding="utf-8"
)

# -------------------------
# 5. ZIP 생성
# -------------------------

zip_path = shutil.make_archive(
    "/content/evac_model_round3_auto_final",
    "zip",
    root_dir=OUT
)

print("\n==============================")
print("✅ Round 3 AUTO 최종 내보내기 완료")
print("모델:", BEST_ROUND3)
print("폴더:", OUT)
print("ZIP:", zip_path)
print("==============================")

# -------------------------
# 6. 다운로드
# -------------------------

files.download(zip_path)